# Observable class

The `Observable` class loads compressed estimators data (see `acm.estimators`), and the associated model.
It provides methods to access the data and model predictions, with the ability to filter the data based on specific criteria.

In this notrebook, we will showcase the features of the class, without looking at the implementation details. The different implementations specificities are available in the notebooks beside this one.

In [ ]:
# Setup
import numpy as np
from helpers import make_file

from acm import setup_logging

setup_logging()

backend = "lsstypes"  # Try it with type="xarray" as well!
data = make_file("observable.h5", backend=backend)

## The Observable factory

The `Observable` class is designed to be used as a factory, which means that it can create instances of different types of observables based on the input parameters. The factory method takes in the name of the observable backend and returns an instance of the corresponding class.

There are currently two backends, corresponding to two different compression/storage types: `xarray` and `lsstypes`.

You can either pass the loaded data directly to the factory with the correct backend name, or you can pass the path to the data file and the factory will determine the correct backend and load the observable data for you.

In [ ]:
from acm.observables import Observable
from acm.observables.lsstypes import LsstypesObservable  # For typing

# Creating an observable via the factory — explicit backend
obs = Observable(backend=backend, data=data)

# Loading from a file — backend auto-detected
obs: LsstypesObservable = Observable.load("observable.h5")

# This can take a few seconds to display for lsstypes, depending on the number of mocks on which filters are applied.
obs

## Observable properties

The Observable class expects the registered data to contaoin "data elements", which are registered in the data trough accessible keys (e.g. DataArrays in Datasets for the xarray implementation). Some common data elements are usually expected to be present in the data:
- `x`: the input data, usually a 2D array of shape (n_samples, n_params)
- `y`: the output data, usually a 2D array of shape (n_samples, n_features)
- `covariance_y`: the covariance array, used to estimate the data error, of shape (n_mocks, n_features)

> The prediction errors are estimated by also using `x_test` and `y_test` data elements, which are subsets of `x` and `y`, respectively.

While the `Observable` class calls return 2D arrays by default, some functions expose parameters that allow some control over trhe shape or type of their outputs:
- `raw=True` will return the raw data object, in its default type (e.g. DataArray in the xarray implementation), without any filtering
- `nested=True` will return a nested array instead of a 2D array. The shape of that nested array depends on the implementation.  

### Filters and selection

The `Observable` class provides methods to select specific data elements from the loaded data. Those selections are stored in the instance state, and applied only when the data is accessed. This allows to keep the original data intact, and to apply different selections on the same data.

**Filtering** allows the selection of a subset of the data based on specific criteria. Filters allow selections from single values, list/tuples of values, or `slices` of values.

> NOTE: It is recommended to select singles values trough lists to conserve the data structure and avoid unexpected behavior.

In [ ]:
# Filtering the data — Allows a mix of scalar, list and slice of values
obs.set_filters(i=[0], ell=[0, 2], k=slice(0, 10))
obs

In [ ]:
# Retrieving data as flattened 2D arrays (ready for a model / MCMC)
x = obs.get_data("x")
y = obs.get_data("y")
x.shape, y.shape

**Selection** allows the selection of specific indices on the 2D-flattened data. To ensure that the selection is applied to the correct data elements, the set_selection method requires to also precise the names of the data elements to apply the selection on. 

This name is preserved, and only data elements matching the exact name or the name prefixed by `"like_"` will have selection applied.
For example, predictions or the covariance array (used to compute the covariance matrix) are selected as `"like_y"`.

In [ ]:
# Selecting specific features on top of the filters - Only for 2D arrays!
obs.set_selection("y", indices=[0, 1, 2, 5, 10])
s1 = obs.get_data("y").shape # 2D array - selection applied
s2 = obs.get_data("y", nested=True).shape # Nested array - selection not applied
s1, s2

In [ ]:
# Access the raw data without any filters or selections applied
obs.get_data("y", raw=True).shape

In [ ]:
# Clearing filters and selection
obs.clear_filters()
obs

### Other properties

The `Observable` class also exposes other methods and properties for convenience.

When the `x` data element is present, it is possible to access the ordered parameter names through the `x_names` property.

In [ ]:
obs.x_names

It is also possible to access the covariance matrix of the data trough the `get_covariance_matrix` method, which will compute the covariance matrix from the covariance array (`covariance_y`) if it is present in the data.

In [ ]:
# Data covariance matrix
obs.set_filters(i=0, ell=[0, 2])
cov = obs.get_covariance_matrix(volume_factor=64, prefactor=1.0)
cov.shape

The `get_handle` method allows to access the a string representation of the registered filters and selections, which can be useful to create a unique identifier of the selection (like a filename).

To mitigate long filenames (especially if combined to other identifiers), the `get_handle` method can also return a hash of the handle string, which can be used as a unique identifier.

A prefix can be added to the handle string or hash, to create a more readable identifier.

In [ ]:
# A unique, filename-safe handle describing the current filtering
h1 = obs.get_handle()

# We can prefix the handle with a name for clarity
h2 = obs.get_handle(prefix="pk")

# Also handles long filename cases with a maximum length (not including prefix)
h3 = obs.get_handle(prefix="pk", hlength=10) # After 10 characters, the handle is hashed

h1, h2, h3

It is also possible to copy or deepcopy an `Observable` instance, which will create a new instance with the same data and selections. This can be useful to create different selections on the same data without modifying the original instance.

In [ ]:
# Copying an observable — filters travel independently on each copy
import copy

obs_copy = copy.deepcopy(obs)
obs_copy.set_filters(i=1)

obs.filters, obs_copy.filters

## ObservableModel

The `Observable` class also provides access to a `sunbird`-based model - usually the one trained from the data contained in the `Observable` instance. The model can be accessed through the `model` property. It can be registered at the class initalization by passing the `model` argument, or it can be set later in the `model` attribute (otherwise set at `None`).

The `ObservableModel` class provides a wrapper around the `sunbird` model for call consistency.

In [ ]:
# Attaching a trained model
from acm.observables import ObservableModel

model = ObservableModel.load("checkpoints/pk_lin.ckpt") # Not available in this example
obs.model = model

obs # Now has a model

The predictions (inference) of the model can be accessed trough the `get_prediction` method of the `Observable` class, which will return the predictions for the selected data elements, and apply the registered filters and selections.

In [ ]:
# Model predictions — automatically matches the observable's filtering
x_query = obs.get_data("x")[:5] # 5 first mocks coordinates
pred = obs.get_prediction(x_query)
pred.shape

The model error is estimated from the difference between the prediction and true values of a test set, accessed trough the `x_test` and `y_test` data elements. The error can be accessed trough the `get_model_error` method of the `Observable` class, which will return the error for the selected data elements, and apply the registered filters and selections.

In [ ]:
# Model error and covariance, evaluated against the observable's test set
error = obs.get_model_error(method="median")
model_cov = obs.get_model_covariance(prefactor=1.0, method="mad", diag=False)

error.shape, model_cov.shape

Finally, the model allows to register a correction on the raw predictions (a callable that takes in a 2D array of predictions and returns a 2D array of corrected predictions), which can be used to correct for systematic errors in the model. The correction can be registered through the `transform` model attribute, or at the model initialization. The correction is applied to the predictions when the `get_prediction` method is called from the model.

In [ ]:
# Optional post-processing applied to every model prediction
def correction(pred: np.ndarray) -> np.ndarray:  # noqa: D103
    return pred * 0.5  # e.g. a phase correction

model.transform = correction
obs.get_prediction(x_query).shape